<a href="https://colab.research.google.com/github/Dream-blue-wasTaken/PMPP/blob/main/vector_add/PMPP_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#date is 27 09 2026 , doin chapter 2
!nvidia-smi

Sun Sep 27 20:31:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/drive/MyDrive/CUDA/PM​​PP

In [ ]:
%%writefile vector_add.cpp

#include <iostream>

void vecAdd(float* A_h, float* B_h, float* C_h, int n) {
    for (int i = 0; i < n; ++i) {
        C_h[i] = A_h[i] + B_h[i];
    }
}

int main() {
    int N = 10;

    float A[N];
    float B[N];
    float C[N];

    // Initialize A and B
    for (int i = 0; i < N; i++) {
        A[i] = i;
        B[i] = 2 * i;
    }

    // Vector addition
    vecAdd(A, B, C, N);

    // Print result
    for (int i = 0; i < N; i++) {
        std::cout << A[i] << " + "
                  << B[i] << " = "
                  << C[i] << std::endl;
    }

    return 0;
}

Writing vector_add.cpp


In [ ]:
!g++ vector_add.cpp -o vector_add

In [ ]:
!./vector_add

0 + 0 = 0
1 + 2 = 3
2 + 4 = 6
3 + 6 = 9
4 + 8 = 12
5 + 10 = 15
6 + 12 = 18
7 + 14 = 21
8 + 16 = 24
9 + 18 = 27


In [2]:
%%writefile vector_add.cpp

#include <iostream>

void vecAdd(float* A, float* B, float* C, int n) {
    int size = n * sizeof(float);
    float *d_a , *d_b , *d_c ;

}

int main() {
    int N = 10;

    float A[N];
    float B[N];
    float C[N];

    // Initialize A and B
    for (int i = 0; i < N; i++) {
        A[i] = i;
        B[i] = 2 * i;
    }

    // Vector addition
    vecAdd(A, B, C, N);

    // Print result
    for (int i = 0; i < N; i++) {
        std::cout << A[i] << " + "
                  << B[i] << " = "
                  << C[i] << std::endl;
    }

    return 0;
}

Writing vector_add.cpp


In [13]:
%%writefile vector_add.cu

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

__global__ void vecAddKernal(float *A_d , float *B_d , float *C_d , int n){
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i< n){
      C_d[i] = A_d[i] + B_d[i];
  }
}

void vecAdd(float *A , float *B , float *C , int n){
    float *A_d ,  *B_d , *C_d;
    int size = n * sizeof(float);

    cudaMalloc((void **)&A_d , size);
    cudaMalloc((void **)&B_d , size);
    cudaMalloc((void **)&C_d , size);

    cudaMemcpy(A_d , A , size , cudaMemcpyHostToDevice);
    cudaMemcpy(B_d , B , size , cudaMemcpyHostToDevice);

    vecAddKernal<<<ceil(n/256.0) , 256>>>(A_d , B_d , C_d , n);

    cudaMemcpy(C,C_d , size , cudaMemcpyDeviceToHost);

    cudaFree(A_d);
    cudaFree(B_d);
    cudaFree(C_d);
}
int main(){
    int n = 1000;
    int size = n * sizeof(float);


    float *A = (float *)malloc(size);
    float *B = (float *)malloc(size);
    float *C = (float *)malloc(size);


    for (int i = 0; i < n; i++)
    {
        A[i] = i;
        B[i] = 2 * i;
    }
    vecAdd(A, B, C, n);
    for (int i = 0; i < 10; i++)
    {
        printf("C[%d] = %.1f\n", i, C[i]);
    }


    free(A);
    free(B);
    free(C);

    return 0;
}




Overwriting vector_add.cu


In [14]:
!nvcc vector_add.cu -o vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [15]:
!./vector_add

C[0] = 0.0
C[1] = 3.0
C[2] = 6.0
C[3] = 9.0
C[4] = 12.0
C[5] = 15.0
C[6] = 18.0
C[7] = 21.0
C[8] = 24.0
C[9] = 27.0
